# Information Retrieval with BM25 and Embeddings
by Mickey Choy and Yuheng Ouyang

## 0. Imports and helper functions

In [38]:
import gzip
import json
import pickle
import re
import string
import pandas as pd
from IPython.display import display, Markdown
from textwrap import shorten

# Get the root
import sys
import os
project_root = os.path.abspath(os.path.join(os.getcwd(), ".."))
sys.path.append(project_root)

# Suppress warnings
import warnings
warnings.filterwarnings('ignore')

#import nltk
#nltk.download("stopwords")
from nltk.corpus import stopwords
from rank_bm25 import BM25Okapi
from langchain_core.documents import Document
from sentence_transformers import SentenceTransformer

# Import scripts 
from src.bm25 import build_bm25, save_bm25, load_bm25, bm25_search
from src.semantic import (
    build_embeddings,
    build_faiss_index,
    save_faiss,
    load_faiss,
    semantic_search
)

In [24]:
def load_jsonl_gz(path, n=200):
    '''
    load first N records from a .jsonl.gz file
    '''
    records = []
    with gzip.open(path, "rt", encoding="utf-8") as f:
        for i, line in enumerate(f):
            if i >= n:
                break
            try:
                records.append(json.loads(line))
            except json.JSONDecodeError:
                continue
    return records

In [36]:
def display_search_results(results, query, system_name, max_chars=200):
    '''
    Pretty display for retrieval results.
    '''
    rows = []

    for rank, r in enumerate(results, start=1):
        metadata = r["metadata"]
        source = metadata["source"]

        raw_text = r["text"].strip()
        parts = [p.strip() for p in raw_text.split("\n\n") if p.strip()]

        title = parts[0]
        body = " ".join(parts[1:]) if len(parts) > 1 else raw_text
        body = " ".join(body.split())

        if source == "metadata":
            categories = metadata.get("categories", [])
            categories_str = " > ".join(categories[:4]) if isinstance(
                categories, list) else ""
            info = metadata.get("main_category", "")
            if categories_str:
                info = f"{info} | {categories_str}" if info else categories_str
        else:
            info = f'rating={metadata.get("rating", "")}, verified={metadata.get("verified_purchase", "")}'

        rows.append({
            "Rank": rank,
            "Score": float(r["score"]),
            "Source": source,
            "ASIN": metadata["asin"],
            "Title": title,
            "Info": info,
            "Snippet": shorten(body, width=max_chars, placeholder="...")
        })

    df = pd.DataFrame(rows)

    title = f"#### {system_name} results for: '{query}'"
    display(Markdown(title))

    display(
        df.style
        .hide(axis="index")
        .format({"Score": "{:.3f}"})
        .set_properties(subset=["Title", "Info", "Snippet"], **{"text-align": "left"})
        .set_properties(subset=["Snippet"], **{"white-space": "pre-wrap"})
    )

    return df

## 1. Data Exploration and Preprocessing

In [6]:
# Load small subsets (200 rows)
meta_path = "../data/raw/meta_Appliances.jsonl.gz"
review_path = "../data/raw/Appliances.jsonl.gz"

meta_records = load_jsonl_gz(meta_path, n=200)
review_records = load_jsonl_gz(review_path, n=200)

print(f"Loaded {len(meta_records)} metadata records")
print(f"Loaded {len(review_records)} review records")

# Convert to DataFrames for easier EDA
meta_df = pd.DataFrame(meta_records)
review_df = pd.DataFrame(review_records)

meta_df.head()

Loaded 200 metadata records
Loaded 200 review records


,main_category,title,average_rating,rating_number,features,description,price,images,videos,store,categories,details,parent_asin,bought_together
0,Industrial & Scientific,"ROVSUN Ice Maker Machine Countertop, Make 44lb...",3.7,61,[【Quick Ice Making】This countertop ice machine...,[],NaN,[{'thumb': 'https://m.media-amazon.com/images/...,[{'title': 'Our Point of View on the Euhomy Ic...,ROVSUN,"[Appliances, Refrigerators, Freezers & Ice Mak...","{'Brand': 'ROVSUN', 'Model Name': 'ICM-2005', ...",B08Z743RRD,None
1,Tools & Home Improvement,"HANSGO Egg Holder for Refrigerator, Deviled Eg...",4.2,75,"[Plastic, Practical Kitchen Storage - Our egg ...",[],NaN,[{'thumb': 'https://m.media-amazon.com/images/...,[{'title': '10 Eggs Egg Holder for Refrigerato...,HANSGO,"[Appliances, Parts & Accessories, Refrigerator...","{'Manufacturer': 'HANSGO', 'Part Number': 'HAN...",B097BQDGHJ,None
2,Tools & Home Improvement,"Clothes Dryer Drum Slide, General Electric, Ho...",3.5,18,[],"[Brand new dryer drum slide, replaces General ...",NaN,[{'thumb': 'https://m.media-amazon.com/images/...,[],GE,"[Appliances, Parts & Accessories]","{'Manufacturer': 'RPI', 'Part Number': 'WE1M33...",B00IN9AGAE,None
3,Tools & Home Improvement,154567702 Dishwasher Lower Wash Arm Assembly f...,4.5,26,[MODEL NUMBER:154567702 Dishwasher Lower Wash ...,[MODEL NUMBER:154567702 Dishwasher Lower Wash ...,NaN,[{'thumb': 'https://m.media-amazon.com/images/...,[],folosem,"[Appliances, Parts & Accessories, Dryer Parts ...","{'Manufacturer': 'folosem', 'Part Number': '15...",B0C7K98JZS,None
4,Tools & Home Improvement,Whirlpool W10918546 Igniter,3.8,12,[This is a Genuine OEM Replacement Part.],[Whirlpool Igniter],25.07,[{'thumb': 'https://m.media-amazon.com/images/...,[],Whirlpool,"[Appliances, Parts & Accessories]","{'Manufacturer': 'Whirlpool', 'Part Number': '...",B07QZHQTVJ,None


### 1.1. Dataset overview

In [7]:
print("Metadata fields:", meta_df.columns.tolist())
print("Review fields:", review_df.columns.tolist())
print()
print("Metadata shape:", meta_df.shape)
print("Review shape:", review_df.shape)

Metadata fields: ['main_category', 'title', 'average_rating', 'rating_number', 'features', 'description', 'price', 'images', 'videos', 'store', 'categories', 'details', 'parent_asin', 'bought_together']
Review fields: ['rating', 'title', 'text', 'images', 'asin', 'parent_asin', 'user_id', 'timestamp', 'helpful_vote', 'verified_purchase']

Metadata shape: (200, 14)
Review shape: (200, 10)


### 1.2. Sample records

In [8]:
meta_df.sample(3)
review_df.sample(3)

,rating,title,text,images,asin,parent_asin,user_id,timestamp,helpful_vote,verified_purchase
51,5.0,Great item for easy cleanup.,"These are great, especially when cooking items...",[],B07935N99S,B07935N99S,AGYDZXXV32IH5YIDK5XSEC5UFIEQ,1578262171522,0,True
160,5.0,Same as fridgadare pricey part,Fits perfectly for our fridgadare gallery frid...,[],B07VG6MYSW,B07VG6MYSW,AHGPF6KEWVFUFGEGCOXO3JMSANHQ,1566501240824,1,True
76,5.0,Great for Elderly People!,My elderly father was recently in rehab for a ...,[],B09Q5QP6JP,B09Q5QP6JP,AHV6QCNBJNSGLATP56JAWJ3C4G2A,1658441140130,1,False


### 1.3. Data preprocessing

In [9]:
# Fields we decided to use for retrieval
META_TEXT_FIELDS = ["title", "description", "features"]
REVIEW_TEXT_FIELDS = ["title", "text"]

documents = []

# Process metadata entries
for _, row in meta_df.iterrows():
    text_parts = []

    for field in META_TEXT_FIELDS:
        value = row.get(field)
        if isinstance(value, str) and len(value.strip()) > 0:
            text_parts.append(value.strip())

    # Skip if no meaningful text
    if len(text_parts) == 0:
        continue

    full_text = "\n\n".join(text_parts)

    documents.append(
        Document(
            page_content=full_text,
            metadata={
                "source": "metadata",
                "asin": row.get("parent_asin"),
                "main_category": row.get("main_category"),
                "categories": row.get("categories"),
            }
        )
    )

# Process review entries
for _, row in review_df.iterrows():
    text_parts = []

    for field in REVIEW_TEXT_FIELDS:
        value = row.get(field)
        if isinstance(value, str) and len(value.strip()) > 0:
            text_parts.append(value.strip())

    if len(text_parts) == 0:
        continue

    full_text = "\n\n".join(text_parts)

    documents.append(
        Document(
            page_content=full_text,
            metadata={
                "source": "review",
                "asin": row.get("asin"),
                "rating": row.get("rating"),
                "verified_purchase": row.get("verified_purchase"),
            }
        )
    )

len(documents)

399

#### 1.3.1 Selection and justification of fields for retrieval

For retrieval, we focused on fields that contain meaningful natural‑language text.  
* From the metadata, **`title`**, **`description`**, and sometimes **`features`** provide the most useful product information.
* From the reviews, the key fields are **`text`** (full review body) and **`title`** (short summary).  
Other fields like price, images, or ratings don’t add much value for text‑based retrieval. 


#### 1.3.2. Description of text preprocessing

Light preprocessing was applied to keep the text clean but still semantically rich:  
- all text lowercased  
- extra whitespace and URLs removed 
- punctuation kept
- empty or extremely short entries removed

There is no stemming or lemmatization involved, as modern embedding models handle them well on their own.

## 2. Keyword-based retrieval system (BM25)

In [10]:
bm25, tokenized = build_bm25(documents)
save_bm25(bm25, tokenized, save_dir="../bm25_index/")

bm25, tokenized = load_bm25("../bm25_index/")

### 2.1. Example query results

In [47]:
query = "quiet dishwasher stainless steel"
results = bm25_search(query, bm25, documents, k=5)

display_search_results(results, query=query, system_name="BM25 search");

#### BM25 search results for: 'quiet dishwasher stainless steel'

Rank,Score,Source,ASIN,Title,Info,Snippet
1,11.088,metadata,B0052EK0MW,"KitchenAid Superba Series KUDS30SXSS Fully Integrated Dishwasher, Stainless Steel",Appliances | Appliances > Dishwashers > Built-In Dishwashers,"KitchenAid Superba Series KUDS30SXSS Fully Integrated Dishwasher, Stainless Steel"
2,9.758,metadata,B09XXFX5SK,"MLGB Stainless Steel Brushed Pattern Dishwasher Magnet Cover Panel Decal Home Appliance Art, Stainless Steel Fridge Door Cover Decals Magnetic, Black, Mobile Magnetic 23"" x 26""",Amazon Home | Appliances > Parts & Accessories > Dishwasher Parts & Accessories > Hoses,"MLGB Stainless Steel Brushed Pattern Dishwasher Magnet Cover Panel Decal Home Appliance Art, Stainless Steel Fridge Door Cover Decals Magnetic, Black, Mobile Magnetic 23"" x 26"""
3,7.350,metadata,B00PLD0YUW,"midea HS-209BESS Beer/Beverage Refrigerator and Dispenser, 5.7 Cubic Feet, Stainless Steel","Appliances | Appliances > Refrigerators, Freezers & Ice Makers > Kegerators","midea HS-209BESS Beer/Beverage Refrigerator and Dispenser, 5.7 Cubic Feet, Stainless Steel"
4,7.242,review,B076VPHFH6,stainless steel pitcher,"rating=5.0, verified=True","I actually purchased this to pour one cup of water into my individual cup coffee maker. So I don't froth milk.. but this pitcher is good for my average mug, I purchased a larger one for my larger..."
5,7.143,metadata,B005H47EJ4,"GE CGS985SETSS Cafe 30"" Stainless Steel Gas Sealed Burner Range - Convection","Appliances | Appliances > Ranges, Ovens & Cooktops > Ranges > Freestanding Ranges","GE CGS985SETSS Cafe 30"" Stainless Steel Gas Sealed Burner Range - Convection"


## 3. Semantic-based retrival system

### 3.1. Embeddings with sentence transformers

In [39]:
embeddings, model = build_embeddings(documents)
index = build_faiss_index(embeddings)
save_faiss(index, documents, save_dir="../semantic_index/")

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Batches:   0%|          | 0/13 [00:00<?, ?it/s]

### 3.2. FAISS index and documents

In [14]:
index, documents = load_faiss("../semantic_index/")

### 3.3. Example query results

In [46]:
query = "quiet dishwasher stainless steel"
results = semantic_search(query, index, model, documents, k=5)

display_search_results(results, query=query, system_name="Semantic search");

#### Semantic search results for: 'quiet dishwasher stainless steel'

Rank,Score,Source,ASIN,Title,Info,Snippet
1,0.773,metadata,B0052EK0MW,"KitchenAid Superba Series KUDS30SXSS Fully Integrated Dishwasher, Stainless Steel",Appliances | Appliances > Dishwashers > Built-In Dishwashers,"KitchenAid Superba Series KUDS30SXSS Fully Integrated Dishwasher, Stainless Steel"
2,0.954,review,B09LM3ZW5N,"Easy to clean, look good, love these","rating=5.0, verified=False","Before silicon counter gap covers came out I had thinner Medal ones I had to paint black, and they didn’t clean well!And Those made noise, and the paint would rub off, and I had to paint them..."
3,0.957,metadata,B00T5K1924,Frigidaire Professional FPID2495QF Fully Integrated Dishwasher,Appliances > Dishwashers > Built-In Dishwashers,Frigidaire Professional FPID2495QF Fully Integrated Dishwasher
4,0.968,review,B0047PF5MM,"Finally, the branded product to clean my Miele!","rating=5.0, verified=True",I have been trying to find something that cleans my 6 year old dish washer and in the past it was run the machine empty. Although we have never noticed any deterioration in the wonderful cleaning...
5,0.989,metadata,B01HP14VVU,ERP 809006501 Dishwasher Lower Seal,Tools & Home Improvement | Appliances > Parts & Accessories,ERP 809006501 Dishwasher Lower Seal
